In [1]:
# -----------------------------
# Imports
# -----------------------------
from pathlib import Path
import re
import ast
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

# -----------------------------
# Paths
# -----------------------------
REPO_ROOT = Path("..").resolve()
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

DEMO_SUBSET_PARQUET = DATA_PROCESSED / "demo_jobs_subset.parquet"
DEMO_CLEAN_PARQUET = DATA_PROCESSED / "demo_jobs_clean.parquet"
DEMO_JOB_EMB_NPY = DATA_PROCESSED / "demo_job_emb.npy"

print("Subset parquet exists:", DEMO_SUBSET_PARQUET.exists())
print("Subset parquet path:", DEMO_SUBSET_PARQUET)

Subset parquet exists: True
Subset parquet path: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_jobs_subset.parquet


In [2]:
# -----------------------------
# Load the saved subset from Notebook 8
# -----------------------------
demo_jobs = pd.read_parquet(DEMO_SUBSET_PARQUET)

print("Loaded shape:", demo_jobs.shape)
print("Loaded columns:")
print(demo_jobs.columns.tolist())

display(demo_jobs.head(5))

Loaded shape: (5000, 17)
Loaded columns:
['job_id', 'experience', 'qualifications', 'location', 'salary_range', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_id,experience,qualifications,location,salary_range,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,1031432770967215,4 to 9 Years,M.Com,Vientiane,$55K-$128K,Lao PDR,Full-Time,2022-11-24,Java Developer,Java Software Engineer,Stack Overflow Jobs,"Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software solutions.","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","Java programming Java frameworks (e.g., Spring) Object-oriented design Code debugging Software development Problem-solving skills","Design, code, test, and maintain Java-based software applications. Collaborate with cross-functional teams on software development projects. Debug and resolve software defects and issues.",Deutsche Lufthansa AG,"{""Sector"":""Aviation and Travel"",""Industry"":""Airlines & Aviation"",""City"":""Cologne"",""State"":""North Rhine-Westphalia"",""Zip"":""50667"",""Website"":""https://www.lufthansa.com/de/de/homepage"",""Ticker"":""LHAG"",""CEO"":""Carsten Spohr""}"
1,1622457186976888,2 to 12 Years,PhD,Brasilia,$58K-$119K,Brazil,Contract,2023-04-08,Architectural Designer,Architectural Drafter,Jobs2Careers,Architectural Drafters assist architects and engineers in creating detailed technical drawings and plans for buildings and structures. They use computer-aided design (CAD) software to produce accurate and precise architectural drawings.,"{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Architectural drafting AutoCAD 2D and 3D modeling Blueprint reading Building codes Collaboration with architects Detail-oriented,Prepare detailed architectural drawings and plans using computer-aided design (CAD) software. Assist architects in project documentation and coordination. Ensure compliance with building codes and regulations.,Bank of America,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Charlotte"",""State"":""North Carolina"",""Zip"":""28255"",""Website"":""www.bankofamerica.com"",""Ticker"":""BAC"",""CEO"":""""}"
2,852684152003067,0 to 12 Years,PhD,Algiers,$59K-$109K,Algeria,Contract,2022-05-25,Business Analyst,Healthcare Business Analyst,Idealist,"Healthcare Business Analysts work in the healthcare industry, analyzing data and processes to improve healthcare delivery. They collaborate with healthcare professionals and IT teams to optimize systems and workflows.","{'Childcare Assistance, Paid Time Off (PTO), Relocation Assistance, Flexible Work Arrangements, Professional Development'}",Healthcare industry knowledge Health data analysis HIPAA regulations EMR systems,"Work in the healthcare sector, analyzing healthcare data and processes to improve patient care and operational efficiency. Identify opportunities for process improvement and cost reduction. Collaborate with healthcare professionals and stakeholders.",Walt Disney,"{""Sector"":""Entertainment"",""Industry"":""Entertainment"",""City"":""Burbank"",""State"":""California"",""Zip"":""91521"",""Website"":""www.thewaltdisneycompany.com"",""Ticker"":""DIS"",""CEO"":""Robert A. Chapek""}"
3,697591924412704,5 to 14 Years,BBA,Gaborone,$55K-$92K,Botswana,Full-Time,2023-08-07,QA Analyst,Software QA Tester,SimplyHired,"Software QA Testers ensure the quality of software products by designing and executing test cases, identifying defects, and reporting issues to developers. They play a critical role in ensuring software reliability and functionality.","{'Health Insurance, Retirement Plans, Flexible Work Arrangements, Employee Assistance Programs (EAP), Bonuses and Incentive Programs'}","Software quality assurance Test planning Test case design Test execution Defect tracking Te

In [3]:
# -----------------------------
# Verify that the expected schema exists
# -----------------------------
required_columns = [
    "job_id",
    "experience",
    "qualifications",
    "location",
    "salary_range",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "skills",
    "responsibilities",
    "company_name",
    "company_profile",
]

missing_columns = [c for c in required_columns if c not in demo_jobs.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Schema check passed.")

Schema check passed.


In [4]:
# -----------------------------
# Light text cleaning helper
# -----------------------------
def clean_text(x):
    if pd.isna(x):
        return ""

    s = str(x)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


# -----------------------------
# Normalize skill phrase lightly
#
# Important:
# We do NOT aggressively compress or throw away
# useful multi-word phrases here.
# -----------------------------
def normalize_skill_phrase(skill):
    s = clean_text(skill).lower()
    s = s.strip(" ,;:.[]{}<>|")
    return s


# -----------------------------
# Build dashboard description snippet
# -----------------------------
def build_description_snippet(text, max_chars=220):
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."

In [5]:
# -----------------------------
# Conservative skill parsing
#
# This dataset often stores skills as compressed text, not clean lists.
# So we:
# - keep meaningful multi-word chunks
# - split on strong separators if present
# - otherwise chunk the text into phrase windows
#
# We prefer preserving information over over-cleaning.
# -----------------------------
def chunk_skill_blob(text, chunk_size=3):
    """
    Convert compressed skill text into overlapping 2-3 word phrases.
    This is a fallback for rows where the dataset does not provide
    comma-separated skill lists.
    """
    text = normalize_skill_phrase(text)

    if not text:
        return []

    # Remove bracket wrappers if present
    text = text.strip("[]").strip()

    # Split into word tokens
    tokens = [t for t in re.split(r"\s+", text) if t]

    if not tokens:
        return []

    phrases = []

    # Build 2-word and 3-word phrases
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            phrase = " ".join(tokens[i:i+n]).strip()
            if phrase:
                phrases.append(phrase)

    return phrases


def parse_skills(x):
    """
    Parse the raw skills field into a usable list.

    Strategy:
    1. Try list-like parsing if the field looks like a Python list
    2. Split on commas/semicolons if separators exist
    3. Otherwise treat it as compressed text and create phrase chunks

    We intentionally keep this parser more permissive than the earlier version
    so jobs do not lose all their skills.
    """
    if pd.isna(x):
        return []

    s = str(x).strip()
    if not s:
        return []

    # Try Python-style list parsing first
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                cleaned = [normalize_skill_phrase(v) for v in parsed if normalize_skill_phrase(v)]
                return sorted(set(cleaned))
        except Exception:
            # Fall through to alternative parsing
            pass

    # Split on commas/semicolons if they exist
    if "," in s or ";" in s:
        parts = re.split(r"[;,]", s)
        parts = [normalize_skill_phrase(p) for p in parts if normalize_skill_phrase(p)]

        # Keep meaningful phrases only
        parts = [p for p in parts if len(p.split()) <= 6 and len(p) >= 2]
        return sorted(set(parts))

    # Otherwise treat as compressed text blob
    chunked = chunk_skill_blob(s)

    # Remove very weak chunks
    chunked = [c for c in chunked if len(c) >= 3 and len(c.split()) <= 4]

    # Remove exact duplicates while keeping useful multi-word phrases
    return sorted(set(chunked))

In [6]:
# -----------------------------
# Lightly clean important text columns
# and parse skills into job_skills_list
# -----------------------------
TEXT_COLUMNS = [
    "experience",
    "qualifications",
    "location",
    "salary_range",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "responsibilities",
    "company_name",
    "company_profile",
]

for col in TEXT_COLUMNS:
    demo_jobs[col] = demo_jobs[col].apply(clean_text)

demo_jobs["job_skills_list"] = demo_jobs["skills"].apply(parse_skills)

display(
    demo_jobs[[
        "job_title",
        "role",
        "skills",
        "job_skills_list"
    ]].head(10)
)

,job_title,role,skills,job_skills_list
0,Java Developer,Java Software Engineer,"Java programming Java frameworks (e.g., Spring) Object-oriented design Code debugging Software development Problem-solving skills",[java programming java frameworks (e.g]
1,Architectural Designer,Architectural Drafter,Architectural drafting AutoCAD 2D and 3D modeling Blueprint reading Building codes Collaboration with architects Detail-oriented,"[2d and, 2d and 3d, 3d modeling, 3d modeling blueprint, and 3d, and 3d modeling, architects detail-oriented, architectural drafting, architectural drafting autocad, autocad 2d, autocad 2d and, blueprint reading, blueprint reading building, building codes, building codes collaboration, codes collaboration, codes collaboration with, collaboration with, collaboration with architects, drafting autocad, drafting autocad 2d, modeling blueprint, modeling blueprint reading, reading building, reading building codes, with architects, with architects detail-oriented]"
2,Business Analyst,Healthcare Business Analyst,Healthcare industry knowledge Health data analysis HIPAA regulations EMR systems,"[analysis hipaa, analysis hipaa regulations, data analysis, data analysis hipaa, emr systems, health data, health data analysis, healthcare industry, healthcare industry knowledge, hipaa regulations, hipaa regulations emr, industry knowledge, industry knowledge health, knowledge health, knowledge health data, regulations emr, regulations emr systems]"
3,QA Analyst,Software QA Tester,"Software quality assurance Test planning Test case design Test execution Defect tracking Test automation (e.g., Selenium)",[selenium)]
4,Sales Consultant,Sales Trainer,Sales training Sales coaching Training program development Sales techniques Product knowledge Presentation skills,"[coaching training, coaching training program, development sales, development sales techniques, knowledge presentation, knowledge presentation skills, presentation skills, product knowledge, product knowledge presentation, program development, program development sales, sales coaching, sales coaching training, sales techniques, sales techniques product, sales training, sales training sales, techniques product, techniques product knowledge, training program, training program development, training sales, training sales coaching]"
5,Systems Engineer,Systems Integration Specialist,Systems integration Integration architecture Data mapping Middleware technologies API integration System testing Troubleshooting,"[api integration, api integration system, architecture data, architecture data mapping, data mapping, data mapping middleware, integration architecture, integration architecture data, integration integration, integration integration architecture, integration system, integration system testing, mapping middleware, mapping middleware technologies, middleware technologies, middleware technologies api, system testing, system testing troubleshooting, systems integration, systems integration integration, technologies api, technologies api integration, testing troubleshooting]"
6,Environmental Consultant,Sustainability Consultant,Sustainability consulting Sustainability assessments Sustainable practices Green building standards Environmental policies Client communication,"[assessments sustainable, assessments sustainable practices, building standards, building standards environmental, client communication, consulting sustainability, consulting sustainability assessments, environmental policies, environmental policies client, green building, green building standards, policies client, policies client communication, practices green, practices green building, standards environmental, standards environmental policies, sustainability assessments, sustainability assessments sustainable, sustainability consulting, sustainability consulting sustainability, sustainable practices, sustainable practices green]"
7,Web Developer,Frontend Web Developer,"HTML, CSS, JavaScript Frontend framewor

In [7]:
# -----------------------------
# Check how many rows ended up with empty skill lists
# -----------------------------
empty_skill_count = demo_jobs["job_skills_list"].apply(lambda x: len(x) == 0).sum()
total_rows = len(demo_jobs)

print("Rows with empty job_skills_list:", int(empty_skill_count))
print("Total rows:", total_rows)
print("Empty skill rate:", round(empty_skill_count / total_rows, 4))

display(
    demo_jobs.loc[
        demo_jobs["job_skills_list"].apply(lambda x: len(x) == 0),
        ["job_title", "role", "skills", "job_description"]
    ].head(10)
)

Rows with empty job_skills_list: 79
Total rows: 5000
Empty skill rate: 0.0158


,job_title,role,skills,job_description
102,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
162,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
350,Sales Representative,Inside Sales Representative,"Sales prospecting and lead generation Sales presentation and communication CRM software (e.g., Salesforce) Sales negotiation and closing techniques Product knowledge Relationship building","Inside Sales Representatives are responsible for selling products or services to customers over the phone or through online channels. They engage with leads, answer customer inquiries, and use persuasive communication to close sales and meet revenue targets."
436,Sales Representative,Inside Sales Representative,"Sales prospecting and lead generation Sales presentation and communication CRM software (e.g., Salesforce) Sales negotiation and closing techniques Product knowledge Relationship building","Inside Sales Representatives are responsible for selling products or services to customers over the phone or through online channels. They engage with leads, answer customer inquiries, and use persuasive communication to close sales and meet revenue targets."
700,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
753,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
784,Sales Representative,Inside Sales Representative,"Sales prospecting and lead generation Sales presentation and communication CRM software (e.g., Salesforce) Sales negotiation and closing techniques Product knowledge Relationship building","Inside Sales Representatives are responsible for selling products or services to customers over the phone or through online channels. They engage with leads, answer customer inquiries, and use persuasive communication to close sales and meet revenue targets."
792,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
891,Network Analyst,Network Performance Analyst,"Network performance analysis Network monitoring tools (e.g., Wireshark) Troubleshooting Capacity planning Data analysis Network protocols","Network Performance Analysts monitor and optimize network performance. They collect and analyze network data, identify issues, and implement solutions to enhance network speed, reliability, and efficiency."
1011,Network Analyst,Network Performance Analyst,"Network performance ana

In [8]:
# -----------------------------
# Building semantic text representation for each job
#
# Preserving BOTH:
# - role
# - job_title
#
# Also include skills, but ranking is still supported heavily
# by descriptions/responsibilities even if skills are imperfect.
# -----------------------------
def build_demo_job_text(row):
    skills_str = ", ".join(row["job_skills_list"])

    parts = [
        f"Role: {row['role']}.",
        f"Job title: {row['job_title']}.",
        f"Company: {row['company_name']}.",
        f"Location: {row['location']}, {row['country']}.",
        f"Salary range: {row['salary_range']}.",
        f"Work type: {row['work_type']}.",
        f"Experience required: {row['experience']}.",
        f"Qualifications: {row['qualifications']}.",
        f"Skills: {skills_str}.",
        f"Job description: {row['job_description']}.",
        f"Responsibilities: {row['responsibilities']}.",
        f"Benefits: {row['benefits']}.",
        f"Company profile: {row['company_profile']}.",
        f"Job portal: {row['job_portal']}.",
    ]

    text = " ".join([clean_text(p) for p in parts if clean_text(p)])
    return clean_text(text)

demo_jobs["job_text"] = demo_jobs.apply(build_demo_job_text, axis=1)
demo_jobs["description_snippet"] = demo_jobs["job_description"].apply(build_description_snippet)

display(
    demo_jobs[[
        "role",
        "job_title",
        "company_name",
        "job_skills_list",
        "description_snippet"
    ]].head(10)
)

,role,job_title,company_name,job_skills_list,description_snippet
0,Java Software Engineer,Java Developer,Deutsche Lufthansa AG,[java programming java frameworks (e.g],"Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software so..."
1,Architectural Drafter,Architectural Designer,Bank of America,"[2d and, 2d and 3d, 3d modeling, 3d modeling blueprint, and 3d, and 3d modeling, architects detail-oriented, architectural drafting, architectural drafting autocad, autocad 2d, autocad 2d and, blueprint reading, blueprint reading building, building codes, building codes collaboration, codes collaboration, codes collaboration with, collaboration with, collaboration with architects, drafting autocad, drafting autocad 2d, modeling blueprint, modeling blueprint reading, reading building, reading building codes, with architects, with architects detail-oriented]",Architectural Drafters assist architects and engineers in creating detailed technical drawings and plans for buildings and structures. They use computer-aided design (CAD) software to produce accurate and precise archite...
2,Healthcare Business Analyst,Business Analyst,Walt Disney,"[analysis hipaa, analysis hipaa regulations, data analysis, data analysis hipaa, emr systems, health data, health data analysis, healthcare industry, healthcare industry knowledge, hipaa regulations, hipaa regulations emr, industry knowledge, industry knowledge health, knowledge health, knowledge health data, regulations emr, regulations emr systems]","Healthcare Business Analysts work in the healthcare industry, analyzing data and processes to improve healthcare delivery. They collaborate with healthcare professionals and IT teams to optimize systems and workflows."
3,Software QA Tester,QA Analyst,Best Buy,[selenium)],"Software QA Testers ensure the quality of software products by designing and executing test cases, identifying defects, and reporting issues to developers. They play a critical role in ensuring software reliability and f..."
4,Sales Trainer,Sales Consultant,Sonoco Products,"[coaching training, coaching training program, development sales, development sales techniques, knowledge presentation, knowledge presentation skills, presentation skills, product knowledge, product knowledge presentation, program development, program development sales, sales coaching, sales coaching training, sales techniques, sales techniques product, sales training, sales training sales, techniques product, techniques product knowledge, training program, training program development, training sales, training sales coaching]","Sales Trainers develop and deliver training programs to sales teams. They teach sales techniques, product knowledge, and communication skills to improve the performance and effectiveness of sales representatives."
5,Systems Integration Specialist,Systems Engineer,DaVita,"[api integration, api integration system, architecture data, architecture data mapping, data mapping, data mapping middleware, integration architecture, integration architecture data, integration integration, integration integration architecture, integration system, integration system testing, mapping middleware, mapping middleware technologies, middleware technologies, middleware technologies api, system testing, system testing troubleshooting, systems integration, systems integration integration, technologies api, technologies api integration, testing troubleshooting]",The role of a Systems Integration Specialist involves integrating various software and hardware components to create cohesive systems. You will design and implement solutions that enable different systems to communicate...
6,Sustainability Consultant,Environmental Consultant,Graphic Packaging Holding,"[assessments sustainable, assessments sustainable practices, building standards, building standards envi

In [9]:
# -----------------------------
# Keeping final processed fields for the demo corpus
# -----------------------------
FINAL_COLUMNS = [
    "job_id",
    "job_title",
    "role",
    "salary_range",
    "company_name",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_portal",
    "experience",
    "qualifications",
    "job_description",
    "description_snippet",
    "benefits",
    "responsibilities",
    "company_profile",
    "job_skills_list",
    "job_text",
]

demo_jobs_clean = demo_jobs[FINAL_COLUMNS].copy()

print("Processed demo jobs shape:", demo_jobs_clean.shape)
print("Processed columns:")
print(demo_jobs_clean.columns.tolist())

display(demo_jobs_clean.head(5))

Processed demo jobs shape: (5000, 19)
Processed columns:
['job_id', 'job_title', 'role', 'salary_range', 'company_name', 'location', 'country', 'work_type', 'job_posting_date', 'job_portal', 'experience', 'qualifications', 'job_description', 'description_snippet', 'benefits', 'responsibilities', 'company_profile', 'job_skills_list', 'job_text']


,job_id,job_title,role,salary_range,company_name,location,country,work_type,job_posting_date,job_portal,experience,qualifications,job_description,description_snippet,benefits,responsibilities,company_profile,job_skills_list,job_text
0,1031432770967215,Java Developer,Java Software Engineer,$55K-$128K,Deutsche Lufthansa AG,Vientiane,Lao PDR,Full-Time,2022-11-24,Stack Overflow Jobs,4 to 9 Years,M.Com,"Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software solutions.","Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software so...","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","Design, code, test, and maintain Java-based software applications. Collaborate with cross-functional teams on software development projects. Debug and resolve software defects and issues.","{""Sector"":""Aviation and Travel"",""Industry"":""Airlines & Aviation"",""City"":""Cologne"",""State"":""North Rhine-Westphalia"",""Zip"":""50667"",""Website"":""https://www.lufthansa.com/de/de/homepage"",""Ticker"":""LHAG"",""CEO"":""Carsten Spohr""}",[java programming java frameworks (e.g],"Role: Java Software Engineer. Job title: Java Developer. Company: Deutsche Lufthansa AG. Location: Vientiane, Lao PDR. Salary range: $55K-$128K. Work type: Full-Time. Experience required: 4 to 9 Years. Qualifications: M.Com. Skills: java programming java frameworks (e.g. Job description: Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software solutions.. Responsibilities: Design, code, test, and maintain Java-based software applications. Collaborate with cross-functional teams on software development projects. Debug and resolve software defects and issues.. Benefits: {'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}. Company profile: {""Sector"":""Aviation and Travel"",""Industry"":""Airlines & Aviation"",""City"":""Cologne"",""State"":""North Rhine-Westphalia"",""Zip"":""50667"",""Website"":""https://www.lufthansa.com/de/de/homepage"",""Ticker"":""LHAG"",""CEO"":""Carsten Spohr""}. Job portal: Stack Overflow Jobs."
1,1622457186976888,Architectural Designer,Architectural Drafter,$58K-$119K,Bank of America,Brasilia,Brazil,Contract,2023-04-08,Jobs2Careers,2 to 12 Years,PhD,Architectural Drafters assist architects and engineers in creating detailed technical drawings and plans for buildings and structures. They use computer-aided design (CAD) software to produce accurate and precise architectural drawings.,Architectural Drafters assist architects and engineers in creating detailed technical drawings and plans for buildings and structures. They use computer-aided design (CAD) software to produce accurate and precise archite...,"{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Prepare detailed architectural drawings and plans using computer-aided design (CAD) software. Assist architects in project documentation and coordination. Ensure compliance with building codes and regulations.,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Charlotte"",""State"":""North Carolina"",""Zip"":""28255"",""Website"":""www.bankofamerica.com"",""Ticker"":""BAC"",""CEO"":""""}","[2d and, 2d and 3d, 3d modeling, 3d modeling blueprint, and 3d, and 3d modeling, architects detail-oriented, architectural drafting, architectural drafting autocad, autocad 2d, autocad 2d and, blueprint re

In [10]:
# -----------------------------
# Save processed job corpus
# -----------------------------
demo_jobs_clean.to_parquet(DEMO_CLEAN_PARQUET, index=False)

print("Saved processed jobs to:", DEMO_CLEAN_PARQUET)

Saved processed jobs to: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_jobs_clean.parquet


In [11]:
# -----------------------------
# Load embedding model
# -----------------------------
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embed_model = SentenceTransformer(MODEL_NAME)

print("Loaded model:", MODEL_NAME)

C:\Users\COMPUTER CARE\anaconda3\envs\jobplatform\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded model: sentence-transformers/all-MiniLM-L6-v2


In [12]:
# -----------------------------
# Generating job embeddings for the demo corpus
# -----------------------------
demo_job_emb = embed_model.encode(
    demo_jobs_clean["job_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

demo_job_emb = np.asarray(demo_job_emb, dtype=np.float32)

print("Embeddings shape:", demo_job_emb.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


In [13]:
# -----------------------------
# Save embeddings for FastAPI
# -----------------------------
np.save(DEMO_JOB_EMB_NPY, demo_job_emb)

print("Saved embeddings to:", DEMO_JOB_EMB_NPY)

Saved embeddings to: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_job_emb.npy


In [14]:
# -----------------------------
# Inspecting a few processed rows to make sure the output looks good
# -----------------------------
display(
    demo_jobs_clean[[
        "job_title",
        "role",
        "company_name",
        "location",
        "work_type",
        "experience",
        "job_portal",
        "job_skills_list",
        "description_snippet"
    ]].head(50)
)

,job_title,role,company_name,location,work_type,experience,job_portal,job_skills_list,description_snippet
0,Java Developer,Java Software Engineer,Deutsche Lufthansa AG,Vientiane,Full-Time,4 to 9 Years,Stack Overflow Jobs,[java programming java frameworks (e.g],"Java Software Engineers develop and maintain software applications using the Java programming language. They write code, debug applications, and collaborate with cross-functional teams to deliver high-quality software so..."
1,Architectural Designer,Architectural Drafter,Bank of America,Brasilia,Contract,2 to 12 Years,Jobs2Careers,"[2d and, 2d and 3d, 3d modeling, 3d modeling blueprint, and 3d, and 3d modeling, architects detail-oriented, architectural drafting, architectural drafting autocad, autocad 2d, autocad 2d and, blueprint reading, blueprint reading building, building codes, building codes collaboration, codes collaboration, codes collaboration with, collaboration with, collaboration with architects, drafting autocad, drafting autocad 2d, modeling blueprint, modeling blueprint reading, reading building, reading building codes, with architects, with architects detail-oriented]",Architectural Drafters assist architects and engineers in creating detailed technical drawings and plans for buildings and structures. They use computer-aided design (CAD) software to produce accurate and precise archite...
2,Business Analyst,Healthcare Business Analyst,Walt Disney,Algiers,Contract,0 to 12 Years,Idealist,"[analysis hipaa, analysis hipaa regulations, data analysis, data analysis hipaa, emr systems, health data, health data analysis, healthcare industry, healthcare industry knowledge, hipaa regulations, hipaa regulations emr, industry knowledge, industry knowledge health, knowledge health, knowledge health data, regulations emr, regulations emr systems]","Healthcare Business Analysts work in the healthcare industry, analyzing data and processes to improve healthcare delivery. They collaborate with healthcare professionals and IT teams to optimize systems and workflows."
3,QA Analyst,Software QA Tester,Best Buy,Gaborone,Full-Time,5 to 14 Years,SimplyHired,[selenium)],"Software QA Testers ensure the quality of software products by designing and executing test cases, identifying defects, and reporting issues to developers. They play a critical role in ensuring software reliability and f..."
4,Sales Consultant,Sales Trainer,Sonoco Products,"Washington, D.C.",Part-Time,2 to 11 Years,Snagajob,"[coaching training, coaching training program, development sales, development sales techniques, knowledge presentation, knowledge presentation skills, presentation skills, product knowledge, product knowledge presentation, program development, program development sales, sales coaching, sales coaching training, sales techniques, sales techniques product, sales training, sales training sales, techniques product, techniques product knowledge, training program, training program development, training sales, training sales coaching]","Sales Trainers develop and deliver training programs to sales teams. They teach sales techniques, product knowledge, and communication skills to improve the performance and effectiveness of sales representatives."
5,Systems Engineer,Systems Integration Specialist,DaVita,Nairobi,Part-Time,5 to 14 Years,Stack Overflow Jobs,"[api integration, api integration system, architecture data, architecture data mapping, data mapping, data mapping middleware, integration architecture, integration architecture data, integration integration, integration integration architecture, integration system, integration system testing, mapping middleware, mapping middleware technologies, middleware technologies, middleware technologies api, system testing, system testing troubleshooting, systems integration, systems integration integration, technologies api, technologies api integration, testing troubleshooting]",The role of a Systems Integration Specialist involves integrating various softw

In [15]:
# -----------------------------
# Verify saved parquet + embeddings
# -----------------------------
print("Processed parquet exists:", DEMO_CLEAN_PARQUET.exists())
print("Embedding file exists:", DEMO_JOB_EMB_NPY.exists())

check_jobs = pd.read_parquet(DEMO_CLEAN_PARQUET)

print("Reloaded processed shape:", check_jobs.shape)
print("Reloaded columns:")
print(check_jobs.columns.tolist())

display(check_jobs[["role", "job_title", "job_skills_list"]].head(10))

Processed parquet exists: True
Embedding file exists: True
Reloaded processed shape: (5000, 19)
Reloaded columns:
['job_id', 'job_title', 'role', 'salary_range', 'company_name', 'location', 'country', 'work_type', 'job_posting_date', 'job_portal', 'experience', 'qualifications', 'job_description', 'description_snippet', 'benefits', 'responsibilities', 'company_profile', 'job_skills_list', 'job_text']


,role,job_title,job_skills_list
0,Java Software Engineer,Java Developer,[java programming java frameworks (e.g]
1,Architectural Drafter,Architectural Designer,"[2d and, 2d and 3d, 3d modeling, 3d modeling blueprint, and 3d, and 3d modeling, architects detail-oriented, architectural drafting, architectural drafting autocad, autocad 2d, autocad 2d and, blueprint reading, blueprint reading building, building codes, building codes collaboration, codes collaboration, codes collaboration with, collaboration with, collaboration with architects, drafting autocad, drafting autocad 2d, modeling blueprint, modeling blueprint reading, reading building, reading building codes, with architects, with architects detail-oriented]"
2,Healthcare Business Analyst,Business Analyst,"[analysis hipaa, analysis hipaa regulations, data analysis, data analysis hipaa, emr systems, health data, health data analysis, healthcare industry, healthcare industry knowledge, hipaa regulations, hipaa regulations emr, industry knowledge, industry knowledge health, knowledge health, knowledge health data, regulations emr, regulations emr systems]"
3,Software QA Tester,QA Analyst,[selenium)]
4,Sales Trainer,Sales Consultant,"[coaching training, coaching training program, development sales, development sales techniques, knowledge presentation, knowledge presentation skills, presentation skills, product knowledge, product knowledge presentation, program development, program development sales, sales coaching, sales coaching training, sales techniques, sales techniques product, sales training, sales training sales, techniques product, techniques product knowledge, training program, training program development, training sales, training sales coaching]"
5,Systems Integration Specialist,Systems Engineer,"[api integration, api integration system, architecture data, architecture data mapping, data mapping, data mapping middleware, integration architecture, integration architecture data, integration integration, integration integration architecture, integration system, integration system testing, mapping middleware, mapping middleware technologies, middleware technologies, middleware technologies api, system testing, system testing troubleshooting, systems integration, systems integration integration, technologies api, technologies api integration, testing troubleshooting]"
6,Sustainability Consultant,Environmental Consultant,"[assessments sustainable, assessments sustainable practices, building standards, building standards environmental, client communication, consulting sustainability, consulting sustainability assessments, environmental policies, environmental policies client, green building, green building standards, policies client, policies client communication, practices green, practices green building, standards environmental, standards environmental policies, sustainability assessments, sustainability assessments sustainable, sustainability consulting, sustainability consulting sustainability, sustainable practices, sustainable practices green]"
7,Frontend Web Developer,Web Developer,"[angular) user experience (ux), css, html, javascript frontend frameworks (e.g, react]"
8,Technical Support Specialist,Customer Support Specialist,"[technical troubleshooting customer support tools (e.g, zendesk]"
9,Chemical Engineer,Process Engineer,"[chemical engineering, chemical engineering process, chemical reactions, chemical reactions safety, design chemical, design chemical reactions, engineering process, engineering process design, laboratory techniques, laboratory techniques problem-solving, problem-solving skills, process design, process design chemical, protocols laboratory, protocols laboratory techniques, reactions safety, reactions safety protocols, safety protocols, safety protocols laboratory, techniques problem-solving, techniques problem-solving skills]"


## Demo Job Corpus Preparation and Embedding Summary

This notebook converts the reduced demo job subset into the final recommendation corpus used by the product demo. The pipeline preserves both `role` and `job_title` as distinct fields, parses the raw skills field conservatively into a reusable list representation, and constructs a rich semantic `job_text` field for each job.

The processed dataset is then embedded using the same sentence-transformer model used elsewhere in the project. The saved parquet file and embedding matrix are the final job-side assets loaded by the FastAPI recommendation service.]].head(10))emo.